In [15]:
import pandas as pd

In [24]:
path = "data/Sahih_Al-Bukhari/sahih_al-bukhari_ahadith.utf8.csv"

df = pd.read_csv(path, encoding="utf-8", dtype=str, names=[
    "hadith_number",
    "hadith"])

In [25]:
df.head()

,hadith_number,hadith
0,1,حدثنا الحميدي عبد الله بن الزبير قال حدثنا سف...
1,2,حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هشا...
2,3,حدثنا يحيى بن بكير قال حدثنا الليث عن عقيل عن...
3,4,حدثنا موسى بن إسماعيل قال حدثنا أبو عوانة قال...
4,5,حدثنا عبدان قال أخبرنا عبد الله قال أخبرنا يو...


In [26]:
for index, row in df[:10].iterrows():
    print(f"{row['hadith_number']}: {row['hadith']}\n")

1:  حدثنا الحميدي عبد الله بن الزبير قال حدثنا سفيان قال حدثنا يحيى بن سعيد الأنصاري قال أخبرني محمد بن إبراهيم التيمي أنه سمع علقمة بن وقاص الليثي يقول سمعت عمر بن الخطاب رضي الله عنه على المنبر قال سمعت رسول الله صلى الله عليه وسلم يقول إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه 

2:  حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هشام بن عروة عن أبيه عن عائشة أم المؤمنين رضي الله عنها أن الحارث بن هشام رضي الله عنه سأل رسول الله صلى الله عليه وسلم فقال يا رسول الله كيف يأتيك الوحي فقال رسول الله صلى الله عليه وسلم أحيانا يأتيني مثل صلصلة الجرس وهو أشده علي فيفصم عني وقد وعيت عنه ما قال وأحيانا يتمثل لي الملك رجلا فيكلمني فأعي ما يقول قالت عائشة رضي الله عنها ولقد رأيته ينزل عليه الوحي في اليوم الشديد البرد فيفصم عنه وإن جبينه ليتفصد عرقا 

3:  حدثنا يحيى بن بكير قال حدثنا الليث عن عقيل عن ابن شهاب عن عروة بن الزبير عن عائشة أم المؤمنين أنها قالت أول ما بدئ به رسول الله صلى الله عليه وسلم من الوحي الرؤيا الصالحة في

In [30]:
import re
from typing import List, Dict

def extract_chain(hadith_text: str) -> Dict[str, any]:
    """
    Extract isnad chain from hadith text.

    Returns:
        {
            'chain': [
                {'name': 'narrator', 'method': 'haddathana/akhbarana/an'},
                ...
            ],
            'matn': 'hadith text after chain'
        }
    """

    # Transmission keywords mapping
    keywords = {
        'حدثنا': 'haddathana',
        'حدثني': 'haddathani',
        'أخبرنا': 'akhbarana',
        'أخبرني': 'akhbarani',
        'عن': 'an',
        'سمع': 'samia',
        'سمعت': 'samitu',
    }

    chain = []
    text = hadith_text.strip()

    # Find where the matn starts (after Prophet mention)
    prophet_pattern = r'(?:رسول الله|النبي)\s+صلى الله عليه وسلم\s+(?:قال|يقول)'
    prophet_match = re.search(prophet_pattern, text)

    if prophet_match:
        # Only process isnad part (before matn)
        isnad_text = text[:prophet_match.end()]
        matn = text[prophet_match.end():].strip()
    else:
        isnad_text = text
        matn = ""

    # Split by transmission keywords while keeping delimiters
    # Pattern: capture keyword + everything until next keyword or قال
    parts = re.split(r'\s+(حدثنا|حدثني|أخبرنا|أخبرني|عن|سمع|سمعت)\s+', isnad_text)

    # Process pairs (keyword, name)
    for i in range(1, len(parts), 2):
        if i + 1 < len(parts):
            keyword = parts[i]
            name_part = parts[i + 1]

            # Extract name (stop at قال, أن, or other delimiters)
            name_match = re.match(r'([^قأي]+?)(?:\s+(?:قال|أن|أنه|يقول)|$)', name_part)
            if name_match:
                name = name_match.group(1).strip()

                # Clean common suffixes
                name = re.sub(r'\s+رضي الله عنه.*$', '', name)
                name = re.sub(r'\s+صلى الله عليه وسلم.*$', '', name)
                name = name.strip()

                if name and len(name) > 2:  # Avoid empty or very short matches
                    chain.append({
                        'name': name,
                        'method': keywords.get(keyword, keyword)
                    })

    return {
        'chain': chain,
        'matn': matn
    }

In [31]:
extract_chain(df.iloc[0]['hadith'])

{'chain': [],
 'matn': 'إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه'}

In [32]:
# Extract chains for first 30 hadiths (MVP)
df_mvp = df.head(30).copy()
df_mvp['extracted'] = df_mvp['hadith'].apply(extract_chain)

# See results
for idx, row in df_mvp.head(3).iterrows():
    print(f"\n=== Hadith {row['hadith_number']} ===")
    print(f"Narrators: {len(row['extracted']['chain'])}")
    for n in row['extracted']['chain']:
        print(f"  - {n['name']} ({n['method']})")


=== Hadith 1 ===
Narrators: 0

=== Hadith 2 ===
Narrators: 2
  - مالك (akhbarana)
  - هشام بن عروة (an)

=== Hadith 3 ===
Narrators: 1
  - ابن شهاب (an)


In [33]:
import ollama
import json
from typing import Dict, List

def extract_chain_llm(hadith_text: str, hadith_num: str) -> Dict:
    """Extract isnad chain using local LLM."""

    prompt = f"""Extract the chain of narrators (isnad) from this Arabic hadith.

Hadith text:
{hadith_text}

Return ONLY valid JSON with this exact structure:
{{
    "chain": [
        {{"name": "narrator full name", "method": "haddathana/akhbarana/an/samia"}},
        ...
    ]
}}

Rules:
- Extract ALL narrators in order from the isnad (chain)
- method mapping: حدثنا=haddathana, أخبرنا/أخبرني=akhbarana, عن=an, سمع/سمعت=samia
- Stop before Prophet Muhammad (don't include him as narrator)
- Clean names: remove رضي الله عنه, صلى الله عليه وسلم, قال
- Keep full names like "عبد الله بن الزبير" or "يحيى بن سعيد الأنصاري"

Return ONLY the JSON, no explanation."""

    response = ollama.chat(
        model='qwen2.5:14b-instruct',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0}  # Deterministic
    )

    try:
        content = response['message']['content']
        # Strip markdown code blocks if present
        content = content.replace('```json', '').replace('```', '').strip()
        result = json.loads(content)
        return result
    except Exception as e:
        print(f"Failed to parse for hadith {hadith_num}: {e}")
        print(f"Response: {response['message']['content'][:200]}")
        return {'chain': []}

# Test with first 3 hadiths
def test_llm_extraction():
    import pandas as pd

    df = pd.read_csv(
        "data/Sahih_Al-Bukhari/sahih_al-bukhari_ahadith.utf8.csv",
        encoding="utf-8",
        dtype=str,
        names=["hadith_number", "hadith"]
    )

    for idx, row in df.head(3).iterrows():
        print(f"\n=== Hadith {row['hadith_number']} ===")
        result = extract_chain_llm(row['hadith'], row['hadith_number'])

        print(f"Narrators: {len(result['chain'])}")
        for i, narrator in enumerate(result['chain'], 1):
            print(f"  {i}. {narrator['name']} ({narrator['method']})")

In [34]:
test_llm_extraction()


=== Hadith 1 ===
Narrators: 4
  1. عبد الله بن الزبير (haddathana)
  2. سفيان (haddathana)
  3. يحيى بن سعيد الأنصاري (haddathana)
  4. محمد بن إبراهيم التيمي (akhbarani)

=== Hadith 2 ===
Narrators: 4
  1. عبد الله بن يوسف (haddathana)
  2. مالك (akhbarani)
  3. هشام بن عروة (an)
  4. عروة (an)

=== Hadith 3 ===
Narrators: 6
  1. يحيى بن بكير (haddathana)
  2. الليث (an)
  3. عقيل (an)
  4. ابن شهاب (an)
  5. عروة بن الزبير (an)
  6. عائشة أم المؤمنين (an)


In [35]:
# Extract 30 hadiths
df_mvp = df.head(30).copy()
df_mvp['extracted'] = df_mvp.apply(
    lambda row: extract_chain_llm(row['hadith'], row['hadith_number']),
    axis=1
)

# Save for manual review
df_mvp.to_json('extracted_chains.json', orient='records', force_ascii=False, indent=2)

In [37]:
for index, row in df.iloc[:30].iterrows():
    print(f"{row['hadith_number']}: {row['hadith']}\n")

1:  حدثنا الحميدي عبد الله بن الزبير قال حدثنا سفيان قال حدثنا يحيى بن سعيد الأنصاري قال أخبرني محمد بن إبراهيم التيمي أنه سمع علقمة بن وقاص الليثي يقول سمعت عمر بن الخطاب رضي الله عنه على المنبر قال سمعت رسول الله صلى الله عليه وسلم يقول إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه 

2:  حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هشام بن عروة عن أبيه عن عائشة أم المؤمنين رضي الله عنها أن الحارث بن هشام رضي الله عنه سأل رسول الله صلى الله عليه وسلم فقال يا رسول الله كيف يأتيك الوحي فقال رسول الله صلى الله عليه وسلم أحيانا يأتيني مثل صلصلة الجرس وهو أشده علي فيفصم عني وقد وعيت عنه ما قال وأحيانا يتمثل لي الملك رجلا فيكلمني فأعي ما يقول قالت عائشة رضي الله عنها ولقد رأيته ينزل عليه الوحي في اليوم الشديد البرد فيفصم عنه وإن جبينه ليتفصد عرقا 

3:  حدثنا يحيى بن بكير قال حدثنا الليث عن عقيل عن ابن شهاب عن عروة بن الزبير عن عائشة أم المؤمنين أنها قالت أول ما بدئ به رسول الله صلى الله عليه وسلم من الوحي الرؤيا الصالحة في

# Regex-based Isnad/Matn Splitter

After analyzing the 30 hadiths, I'll create a regex function that identifies where the Prophet's saying (matn) begins and splits it from the chain (isnad).

In [ ]:
import re
from typing import Dict, Optional

def split_isnad_matn(hadith_text: str) -> Dict[str, str]:
    """
    Split hadith into isnad (chain) and matn (Prophet's saying) using regex.

    The function identifies where the Prophet's actual words begin by looking for:
    1. References to the Prophet (رسول الله / النبي)
    2. Followed by صلى الله عليه وسلم
    3. Followed by a speaking verb (قال / يقول / فقال)

    Returns:
        {
            'isnad': 'chain of narrators',
            'matn': 'prophets saying',
            'split_point': 'the marker phrase where split occurred'
        }
    """

    # Patterns that mark the END of isnad and START of matn
    # We don't include trailing \s+ so we don't skip the first character of matn
    matn_patterns = [
        # Pattern 1: "I heard the Messenger of Allah say" - most direct
        r'سمعت\s+رسول الله\s+صلى الله عليه وسلم\s+(?:قال|يقول|فقال)',

        # Pattern 2: "said the Messenger of Allah" (after someone's words)
        r'(?:فقال|قال)\s+رسول الله\s+صلى الله عليه وسلم',

        # Pattern 3: "the Messenger of Allah said"
        r'رسول الله\s+صلى الله عليه وسلم\s+(?:قال|يقول|فقال)',

        # Pattern 4: "the Prophet said"
        r'النبي\s+صلى الله عليه وسلم\s+(?:قال|يقول|فقال)',

        # Pattern 5: "from the Prophet that he said"
        r'عن\s+النبي\s+صلى الله عليه وسلم\s+(?:قال|أنه قال)',

        # Pattern 6: "that the Messenger said"
        r'أن\s+رسول الله\s+صلى الله عليه وسلم\s+(?:قال|يقول)',

        # Pattern 7: More flexible - any mention after narration chain
        # This catches cases where the chain ends with someone saying something about the Prophet
        r'(?:قال|قالت)\s+(?:كان\s+)?رسول الله\s+صلى الله عليه وسلم',

        # Pattern 8: "they asked the Messenger... he said"
        r'(?:قالوا|سألوا)\s+يا\s+رسول الله\s+.*?\s+قال',

        # Pattern 9: "someone asked the Prophet... he said"
        r'(?:سأل|سئل)\s+(?:رجل|رجلا)?\s*(?:رسول الله|النبي)\s+صلى الله عليه وسلم\s+.*?\s+(?:قال|فقال)',

        # Pattern 10: "asked the Messenger... he said" (more specific)
        r'سأل\s+رسول الله\s+صلى الله عليه وسلم\s+.*?\s+(?:قال|فقال)',

        # Pattern 11: Very simple - just "قال" after Prophet mention
        # (use carefully, only if preceded by Prophet reference)
        r'(?:رسول الله|النبي)\s+صلى الله عليه وسلم[^قال]{1,100}?\s+(?:قال|فقال)',
    ]

    text = hadith_text.strip()

    # Try each pattern to find where the matn starts
    best_match = None
    best_position = -1

    for pattern in matn_patterns:
        matches = list(re.finditer(pattern, text))

        if matches:
            # Use the last match (in case the Prophet is mentioned multiple times)
            last_match = matches[-1]

            # Prefer matches that occur later in the text (more likely to be the actual matn start)
            if last_match.end() > best_position:
                best_position = last_match.end()
                best_match = last_match

    if best_match:
        # Split right after the match, but skip any whitespace
        split_pos = best_match.end()
        matn_start = split_pos

        # Skip whitespace after the pattern
        while matn_start < len(text) and text[matn_start].isspace():
            matn_start += 1

        return {
            'isnad': text[:split_pos].strip(),
            'matn': text[matn_start:].strip(),
            'split_point': best_match.group(0).strip(),
            'split_index': split_pos
        }

    # If no clear split found, return the whole text as isnad
    return {
        'isnad': text,
        'matn': '',
        'split_point': 'NOT_FOUND',
        'split_index': len(text)
    }

In [43]:
# Test on first 5 hadiths
print("=" * 100)
for idx, row in df.head(5).iterrows():
    hadith_num = row['hadith_number']
    hadith_text = row['hadith']

    result = split_isnad_matn(hadith_text)

    print(f"\n{'='*100}")
    print(f"HADITH {hadith_num}")
    print(f"{'='*100}")
    print(f"\nSplit marker: {result['split_point'][:80]}...")
    print(f"\n📿 ISNAD (length: {len(result['isnad'])} chars):")
    print(f"{result['isnad'][:200]}...")
    print(f"\n💬 MATN (length: {len(result['matn'])} chars):")
    print(f"{result['matn'][:200]}...")
    print()


HADITH 1

Split marker: سمعت رسول الله صلى الله عليه وسلم يقول...

📿 ISNAD (length: 234 chars):
حدثنا الحميدي عبد الله بن الزبير قال حدثنا سفيان قال حدثنا يحيى بن سعيد الأنصاري قال أخبرني محمد بن إبراهيم التيمي أنه سمع علقمة بن وقاص الليثي يقول سمعت عمر بن الخطاب رضي الله عنه على المنبر قال سمعت...

💬 MATN (length: 117 chars):
إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه...


HADITH 2

Split marker: فقال رسول الله صلى الله عليه وسلم...

📿 ISNAD (length: 229 chars):
حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هشام بن عروة عن أبيه عن عائشة أم المؤمنين رضي الله عنها أن الحارث بن هشام رضي الله عنه سأل رسول الله صلى الله عليه وسلم فقال يا رسول الله كيف يأتيك الوحي فقال...

💬 MATN (length: 227 chars):
أحيانا يأتيني مثل صلصلة الجرس وهو أشده علي فيفصم عني وقد وعيت عنه ما قال وأحيانا يتمثل لي الملك رجلا فيكلمني فأعي ما يقول قالت عائشة رضي الله عنها ولقد رأيته ينزل عليه الوحي في اليوم الشديد البرد فيفص...


HADITH 3

Split m

In [55]:
# Test on ALL 30 hadiths - summary view
print("Testing split_isnad_matn on 30 hadiths:\n")
print(f"{'Hadith':<8} {'Split Found':<12} {'Isnad Len':<12} {'Matn Len':<12} {'Split Marker':<50}")
print("=" * 100)

successful_splits = 0
for idx, row in df.iterrows():
    hadith_num = row['hadith_number']
    hadith_text = row['hadith']

    result = split_isnad_matn(hadith_text)

    split_found = "✓ YES" if result['split_point'] != 'NOT_FOUND' else "✗ NO"
    if result['split_point'] != 'NOT_FOUND':
        successful_splits += 1

    marker = result['split_point'][:45] + "..." if len(result['split_point']) > 45 else result['split_point']

    print(f"{hadith_num:<8} {split_found:<12} {len(result['isnad']):<12} {len(result['matn']):<12} {marker:<50}")

print("=" * 100)
print(f"\nSuccess rate: {successful_splits}/{df.shape[0]} ({successful_splits/df.shape[0]*100:.1f}%)")

Testing split_isnad_matn on 30 hadiths:

Hadith   Split Found  Isnad Len    Matn Len     Split Marker                                      
1        ✓ YES        234          117          سمعت رسول الله صلى الله عليه وسلم يقول            
2        ✓ YES        229          227          فقال رسول الله صلى الله عليه وسلم                 
3        ✓ YES        1439         537          رسول الله صلى الله عليه وسلم أومخرجي هم قال       
4        ✓ YES        185          480          قال كان رسول الله صلى الله عليه وسلم              
5        ✓ YES        218          159          قال كان رسول الله صلى الله عليه وسلم              
6        ✗ NO         4093         0            NOT_FOUND                                         
7        ✓ YES        133          108          قال رسول الله صلى الله عليه وسلم                  
8        ✓ YES        171          46           النبي صلى الله عليه وسلم قال                      
9        ✓ YES        159          267          النبي صلى الله عليه 

In [45]:
# Examine failed cases in detail
failed_hadiths = [6, 10, 11, 14, 25, 26, 27, 28, 29]

print("Examining failed hadiths to find patterns:\n")
for num in failed_hadiths[:3]:  # Look at first 3 failures
    hadith = df.iloc[num-1]['hadith']
    print(f"\n{'='*100}")
    print(f"HADITH {num}:")
    print(f"{'='*100}")
    print(hadith[:500])
    print("\n...")

Examining failed hadiths to find patterns:


HADITH 6:
 حدثنا أبو اليمان الحكم بن نافع قال أخبرنا شعيب عن الزهري قال أخبرني عبيد الله بن عبد الله بن عتبة بن مسعود أن عبد الله بن عباس أخبره أن أبا سفيان بن حرب أخبره أن هرقل أرسل إليه في ركب من قريش وكانوا تجارا بالشأم في المدة التي كان رسول الله صلى الله عليه وسلم ماد فيها أبا سفيان وكفار قريش فأتوه وهم بإيلياء فدعاهم في مجلسه وحوله عظماء الروم ثم دعاهم ودعا بترجمانه فقال أيكم أقرب نسبا بهذا الرجل الذي يزعم أنه نبي فقال أبو سفيان فقلت أنا أقربهم نسبا فقال أدنوه مني وقربوا أصحابه فاجعلوهم عند ظهره ثم 

...

HADITH 10:
 حدثنا سعيد بن يحيى بن سعيد القرشي قال حدثنا أبي قال حدثنا أبو بردة بن عبد الله بن أبي بردة عن أبي بردة عن أبي موسى رضي الله عنه قال قالوا يا رسول الله أي الإسلام أفضل قال من سلم المسلمون من لسانه ويده 

...

HADITH 11:
 حدثنا عمرو بن خالد قال حدثنا الليث عن يزيد عن أبي الخير عن عبد الله بن عمرو رضي الله عنهما أن رجلا سأل النبي صلى الله عليه وسلم أي الإسلام خير قال تطعم الطعام وتقرأ السلام على من عرفت ومن لم تعرف 

...


In [50]:
# Check hadith 1 in detail
result = split_isnad_matn(df.iloc[0]['hadith'])
print("Hadith 1 split result:")
print(f"Isnad length: {len(result['isnad'])}")
print(f"Matn length: {len(result['matn'])}")
print(f"\nMatn content: '{result['matn']}'")
print(f"\nLast 100 chars of isnad: '{result['isnad'][-100:]}'")


Hadith 1 split result:
Isnad length: 234
Matn length: 117

Matn content: 'إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه'

Last 100 chars of isnad: 'اص الليثي يقول سمعت عمر بن الخطاب رضي الله عنه على المنبر قال سمعت رسول الله صلى الله عليه وسلم يقول'


In [52]:
# Look at remaining failures (6, 14, 25, 28, 29)
print("Examining remaining failures:\n")
for num in [14, 25, 28]:
    hadith = df.iloc[num-1]['hadith']
    print(f"\n{'='*100}")
    print(f"HADITH {num}:")
    print(f"{'='*100}")

    # Search for Prophet mentions
    prophet_mentions = re.findall(r'(?:رسول الله|النبي)\s+صلى الله عليه وسلم', hadith)
    print(f"Prophet mentions found: {len(prophet_mentions)}")

    # Show first 400 chars
    print(f"\n{hadith[:400]}")
    print("\n...")

Examining remaining failures:


HADITH 14:
Prophet mentions found: 2

 حدثنا يعقوب بن إبراهيم قال حدثنا ابن علية عن عبد العزيز بن صهيب عن أنس عن النبي صلى الله عليه وسلم ح و حدثنا آدم قال حدثنا شعبة عن قتادة عن أنس قال قال النبي صلى الله عليه وسلم لا يؤمن أحدكم حتى أكون أحب إليه من والده وولده والناس أجمعين 

...

HADITH 25:
Prophet mentions found: 1

 حدثنا أحمد بن يونس وموسى بن إسماعيل قالا حدثنا إبراهيم بن سعد قال حدثنا ابن شهاب عن سعيد بن المسيب عن أبي هريرة أن رسول الله صلى الله عليه وسلم سئل أي العمل أفضل فقال إيمان بالله ورسوله قيل ثم ماذا قال الجهاد في سبيل الله قيل ثم ماذا قال حج مبرور 

...

HADITH 28:
Prophet mentions found: 1

 حدثنا عبد الله بن مسلمة عن مالك عن زيد بن أسلم عن عطاء بن يسار عن ابن عباس قال قال النبي صلى الله عليه وسلم أريت النار فإذا أكثر أهلها النساء يكفرن قيل أيكفرن بالله قال يكفرن العشير ويكفرن الإحسان لو أحسنت إلى إحداهن الدهر ثم رأت منك شيئا قالت ما رأيت منك خيرا قط 

...


## Summary: Regex-based Isnad/Matn Splitter

The `split_isnad_matn()` function successfully splits **25 out of 30 hadiths (83.3%)** into their isnad (chain of narrators) and matn (Prophet's saying) components.

### Key Patterns Used:
1. **سمعت رسول الله صلى الله عليه وسلم يقول** - "I heard the Messenger say"
2. **قال/فقال رسول الله صلى الله عليه وسلم** - "The Messenger said"
3. **النبي صلى الله عليه وسلم قال** - "The Prophet said"
4. **سأل النبي... قال** - "Asked the Prophet... he said"
5. **قالوا يا رسول الله... قال** - "They said O Messenger... he said"

### Limitations:
The function fails on ~17% of hadiths due to:
- Very long complex narratives (Hadith 6 - 4000+ chars)
- Double قال patterns: "قال قال النبي"
- Passive voice: "سئل" without clear قال afterward
- Narrative-style hadiths where the Prophet is quoted indirectly

### Usage:
```python
result = split_isnad_matn(hadith_text)
print(result['isnad'])  # Chain of narrators
print(result['matn'])   # Prophet's saying
```

In [53]:
# Visual example of successful splits
print("🔍 Examples of Successful Isnad/Matn Splits:\n")

for idx in [0, 1, 6, 10]:  # Hadiths 1, 2, 7, 11
    row = df.iloc[idx]
    result = split_isnad_matn(row['hadith'])

    if result['split_point'] != 'NOT_FOUND':
        print(f"\n{'='*100}")
        print(f"📖 HADITH {row['hadith_number']}")
        print(f"{'='*100}")
        print(f"\n🔗 Split Marker: {result['split_point']}")
        print(f"\n📿 ISNAD ({len(result['isnad'])} chars):")
        print(f"   {result['isnad'][:150]}...")
        print(f"\n💬 MATN ({len(result['matn'])} chars):")
        print(f"   {result['matn'][:150]}...")
        print()

🔍 Examples of Successful Isnad/Matn Splits:


📖 HADITH 1

🔗 Split Marker: سمعت رسول الله صلى الله عليه وسلم يقول

📿 ISNAD (234 chars):
   حدثنا الحميدي عبد الله بن الزبير قال حدثنا سفيان قال حدثنا يحيى بن سعيد الأنصاري قال أخبرني محمد بن إبراهيم التيمي أنه سمع علقمة بن وقاص الليثي يقول س...

💬 MATN (117 chars):
   إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه...


📖 HADITH 2

🔗 Split Marker: فقال رسول الله صلى الله عليه وسلم

📿 ISNAD (229 chars):
   حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هشام بن عروة عن أبيه عن عائشة أم المؤمنين رضي الله عنها أن الحارث بن هشام رضي الله عنه سأل رسول الله صلى الل...

💬 MATN (227 chars):
   أحيانا يأتيني مثل صلصلة الجرس وهو أشده علي فيفصم عني وقد وعيت عنه ما قال وأحيانا يتمثل لي الملك رجلا فيكلمني فأعي ما يقول قالت عائشة رضي الله عنها ولق...


📖 HADITH 7

🔗 Split Marker: قال رسول الله صلى الله عليه وسلم

📿 ISNAD (133 chars):
   حدثنا عبيد الله بن موسى قال أخبرنا حنظلة بن أبي سفيان 

In [ ]:
# Apply to all 30 hadiths and create a new column
df_analyzed = df.head(30).copy()
df_analyzed['split_result'] = df_analyzed['hadith'].apply(split_isnad_matn)

# Expand the split_result into separate columns
df_analyzed['isnad'] = df_analyzed['split_result'].apply(lambda x: x['isnad'])
df_analyzed['matn'] = df_analyzed['split_result'].apply(lambda x: x['matn'])
df_analyzed['split_marker'] = df_analyzed['split_result'].apply(lambda x: x['split_point'])

# Display summary statistics
print("📊 Analysis Summary:")
print(f"Total hadiths analyzed: {len(df_analyzed)}")
print(f"Successfully split: {len(df_analyzed[df_analyzed['split_marker'] != 'NOT_FOUND'])}")
print(f"Failed to split: {len(df_analyzed[df_analyzed['split_marker'] == 'NOT_FOUND'])}")
print(f"\nAverage isnad length: {df_analyzed['isnad'].str.len().mean():.0f} characters")
print(f"Average matn length: {df_analyzed[df_analyzed['matn'] != '']['matn'].str.len().mean():.0f} characters")

# Show sample
print("\n📋 First 3 rows:")
df_analyzed[['hadith_number', 'isnad', 'matn']].head(3)

In [70]:
url_test = "https://hadith.islam-db.com/single-book/146/%D8%B5%D8%AD%D9%8A%D8%AD-%D8%A7%D9%84%D8%A8%D8%AE%D8%A7%D8%B1%D9%8A/97663/2"

In [71]:
import requests
from bs4 import BeautifulSoup

text = requests.get(url_test).text
soup = BeautifulSoup(text, 'html.parser')

In [72]:
text

'<!DOCTYPE html>\n<html lang="">\n<head>\n    <meta charset="utf-8">\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\n    <meta name="description" content="    موسوعة الحديث ,أحاديث كتاب صحيح البخاري\n">\n    <meta name="keywords" content="    حَدَّثَنَا &lt;a class=rawy id=5175&gt;عَبْدُ اللَّهِ بْنُ يُوسُفَ &lt;/a&gt; ، قَالَ : أَخْبَرَنَا &lt;a class=rawy id=6659&gt;مَالِكٌ &lt;/a&gt; ، عَنْ &lt;a class=rawy id=805...\n">\n    <meta name="viewport" content="width=device-width, initial-scale=1">\n    <title> موسوعة الحديث    : صحيح البخاري : 2\n    </title>\n    <link rel="stylesheet" href="https://hadith.islam-db.com/dist/css/font.css">\n    <link rel="stylesheet" href="https://hadith.islam-db.com/dist/css/bootstrap.min.css">\n    <link rel="stylesheet" href="https://hadith.islam-db.com/dist/css/uikit.min.css">\n    <link rel="stylesheet" href="https://hadith.islam-db.com/dist/css/rtl.min.css">\n    <link rel="stylesheet" href="https://hadith.islam-db.com/dist/css/flipped

In [ ]:
https://hadith.islam-db.com/books/146/%D8%B5%D8%AD%D9%8A%D8%AD-%D8%A7%D9%84%D8%A8%D8%AE%D8%A7%D8%B1%D9%8A

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from typing import Dict, List

def scrape_hadith(hadith_num: int) -> Dict:
    """Scrape hadith from islam-db.com"""

    # URL pattern - you'll need to figure out the chapter ID (97663 in your example)
    # For now, let's try with direct hadith number
    url = f"https://hadith.islam-db.com/single-book/146/صحيح-البخاري/97663/{hadith_num}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch hadith {hadith_num}: {e}")
        return None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Extract isnad with narrators
    hadith_text = soup.find('p', class_='more-height')
    if not hadith_text:
        return None

    # Extract narrator links from isnad
    narrator_links = hadith_text.find_all('a', class_='rawy')

    chain = []
    for link in narrator_links:
        narrator_id = link.get('id')
        narrator_name = link.text.strip()
        chain.append({
            'name': narrator_name,
            'id': narrator_id
        })

    # Extract narrator details from table
    narrator_table = soup.find('table', class_='table-striped')
    narrator_details = []

    if narrator_table:
        rows = narrator_table.find_all('tr')[1:]  # Skip header
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                name_cell = cols[0]
                link = name_cell.find('a')

                narrator_details.append({
                    'name': link.text.strip() if link else '',
                    'id': link.get('data-id') if link else '',
                    'fame': cols[1].text.strip(),
                    'rank': cols[2].text.strip()
                })

    # Extract matn
    matn_tag = hadith_text.find('a', class_='matn')
    matn = matn_tag.text.strip() if matn_tag else ""

    # Full text
    full_text = hadith_text.get_text(strip=True)

    return {
        'hadith_number': hadith_num,
        'url': url,
        'chain': chain,
        'narrator_details': narrator_details,
        'matn': matn,
        'full_text': full_text
    }

def scrape_bukhari_range(start: int, end: int, output_file: str = 'golden_dataset.json'):
    """Scrape range of hadiths and save"""

    results = []

    for num in range(start, end + 1):
        print(f"Scraping hadith {num}...")
        hadith = scrape_hadith(num)

        if hadith:
            results.append(hadith)
            print(f"  ✓ Got {len(hadith['chain'])} narrators")
        else:
            print(f"  ✗ Failed")

        # Be nice to the server
        time.sleep(1)

    # Save
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Scraped {len(results)} hadiths → {output_file}")

In [74]:
hadith = scrape_hadith(2)

In [ ]:
https://hadith.islam-db.com/narrators/4049/%D8%B9%D8%A7%D8%A6%D8%B4%D8%A9-%D8%A8%D9%86%D8%AA-%D8%B9%D8%A8%D8%AF-%D8%A7%D9%84%D9%84%D9%87-%D8%A8%D9%86-%D8%B9%D8%AB%D9%85%D8%A7%D9%86-%D8%A8%D9%86...

In [75]:
hadith

{'hadith_number': 2,
 'url': 'https://hadith.islam-db.com/single-book/146/صحيح-البخاري/97663/2',
 'chain': [{'name': 'عَبْدُ اللَّهِ بْنُ يُوسُفَ', 'id': '5175'},
  {'name': 'مَالِكٌ', 'id': '6659'},
  {'name': 'هِشَامِ بْنِ عُرْوَةَ', 'id': '8055'},
  {'name': 'أَبِيهِ', 'id': '5594'},
  {'name': 'عَائِشَةَ', 'id': '4049'}],
 'narrator_details': [{'name': 'عَائِشَةَ',
   'id': '4049',
   'fame': 'عائشة بنت أبي بكر الصديق                                                                                                    / توفي في :57',
   'rank': 'صحابي'},
  {'name': 'أَبِيهِ',
   'id': '5594',
   'fame': 'عروة بن الزبير الأسدي                                                                                                    / توفي في :94',
   'rank': 'ثقة فقيه مشهور'},
  {'name': 'هِشَامِ بْنِ عُرْوَةَ',
   'id': '8055',
   'fame': 'هشام بن عروة الأسدي                                                   / ولد في :58                                                    / توفي في :145',
   '

In [13]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

In [14]:
def check_null_names():
    with driver.session() as session:
        result = session.run("""
            MATCH (n:Person)
            WHERE n.name IS NULL
            RETURN n.id, n.fame
            LIMIT 10
        """)

        print("\nPersons with NULL names:")
        for record in result:
            print(f"  ID: {record['n.id']}, Fame: {record['n.fame']}")

check_null_names()


Persons with NULL names:


In [16]:
def debug_hadith_2_edges():
    """Check what edges exist for hadith 2"""

    with driver.session() as session:
        # Check all edges with hadith=2
        result = session.run("""
            MATCH (n1:Person)-[r:NARRATED_FROM {hadith: 2}]->(n2:Person)
            RETURN n1.name as from_narrator,
                   n1.id as from_id,
                   n2.name as to_narrator,
                   n2.id as to_id
            ORDER BY from_narrator
        """)

        print("All NARRATED_FROM edges for hadith 2:")
        for record in result:
            print(f"  {record['from_narrator']} ({record['from_id']}) -> {record['to_narrator']} ({record['to_id']})")

debug_hadith_2_edges()

# Also check what the scraped data says
import json

with open('bukhari_hadiths.json', 'r', encoding='utf-8') as f:
    hadiths = json.load(f)

hadith_2 = next(h for h in hadiths if h['hadith_number'] == 2)

print("\nExpected chain from scraped data:")
for i in range(len(hadith_2['chain']) - 1):
    n1 = hadith_2['chain'][i]
    n2 = hadith_2['chain'][i + 1]
    print(f"  {n1['name']} ({n1['id']}) -> {n2['name']} ({n2['id']})")

All NARRATED_FROM edges for hadith 2:
  عَبْدُ اللَّهِ بْنُ يُوسُفَ (5175) -> مَالِكٌ (6659)
  عُرْوَةَ بْنَ الزُّبَيْرِ (5594) -> عَائِشَةُ (4049)
  مَالِكٌ (6659) -> هِشَامٍ (8055)
  هِشَامٍ (8055) -> عُرْوَةَ بْنَ الزُّبَيْرِ (5594)

Expected chain from scraped data:
  عَبْدُ اللَّهِ بْنُ يُوسُفَ (5175) -> مَالِكٌ (6659)
  مَالِكٌ (6659) -> هِشَامِ بْنِ عُرْوَةَ (8055)
  هِشَامِ بْنِ عُرْوَةَ (8055) -> أَبِيهِ (5594)
  أَبِيهِ (5594) -> عَائِشَةَ (4049)


In [17]:
import json

with open('bukhari_hadiths.json', 'r', encoding='utf-8') as f:
    hadiths = json.load(f)

hadith_2 = next(h for h in hadiths if h['hadith_number'] == 2)
print("\nScraped chain for hadith 2:")
for i, n in enumerate(hadith_2['chain']):
    print(f"  {i+1}. {n['name']} (ID: {n['id']})")


Scraped chain for hadith 2:
  1. عَبْدُ اللَّهِ بْنُ يُوسُفَ (ID: 5175)
  2. مَالِكٌ (ID: 6659)
  3. هِشَامِ بْنِ عُرْوَةَ (ID: 8055)
  4. أَبِيهِ (ID: 5594)
  5. عَائِشَةَ (ID: 4049)


In [18]:
def verify_chain_by_id():
    """Check if chain is complete by following IDs"""

    with driver.session() as session:
        result = session.run("""
            MATCH (h:Hadith {number: 2})-[:HAS_CHAIN]->(first:Person {id: '5175'})
            MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
            WHERE ALL(r IN relationships(path) WHERE r.hadith = 2)
            RETURN [node in nodes(path) | node.id + ':' + node.name] as chain,
                   length(path) as len
            ORDER BY len DESC
            LIMIT 1
        """)

        for record in result:
            print(f"Chain length: {record['len']}")
            print("Chain by ID:name:")
            for node in record['chain']:
                print(f"  -> {node}")

verify_chain_by_id()

Chain length: 4
Chain by ID:name:
  -> 5175:عَبْدُ اللَّهِ بْنُ يُوسُفَ
  -> 6659:مَالِكٌ
  -> 8055:هِشَامٍ
  -> 5594:عُرْوَةَ بْنَ الزُّبَيْرِ
  -> 4049:عَائِشَةُ


In [ ]:
def final_validation_optimized():
    """Validate with memory-efficient queries"""

    with driver.session() as session:

        print("="*60)
        print("FINAL GRAPH VALIDATION")
        print("="*60)

        # Check a sample of chains (not all at once)
        print(f"\n📊 Sample Chain Lengths (first 100 hadiths):")
        result = session.run("""
            MATCH (h:Hadith)-[:HAS_CHAIN]->(first:Person)
            WHERE h.number <= 100
            MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
            WHERE ALL(r IN relationships(path) WHERE r.hadith = h.number)
              AND NOT EXISTS((last)-[:NARRATED_FROM {hadith: h.number}]->())
            RETURN AVG(length(path)) as avg_length,
                   MIN(length(path)) as min_length,
                   MAX(length(path)) as max_length
        """)

        for record in result:
            print(f"   Average: {record['avg_length']:.1f}")
            print(f"   Range: {record['min_length']} - {record['max_length']}")

        # Most common narrators (simpler query)
        print(f"\n👥 Top 10 Narrators by Total Connections:")
        result = session.run("""
            MATCH (n:Person)
            OPTIONAL MATCH (n)-[:NARRATED_FROM]->()
            WITH n, COUNT(*) as out_count
            OPTIONAL MATCH ()-[:NARRATED_FROM]->(n)
            WITH n, out_count, COUNT(*) as in_count
            RETURN n.name, n.rank, (out_count + in_count) as total
            ORDER BY total DESC
            LIMIT 10
        """)

        for record in result:
            print(f"   {record['n.name']}: {record['total']} connections ({record['n.rank']})")

        # Sample chains
        print(f"\n🔗 Sample Chains:")
        for hadith_num in [1, 2, 50, 100]:
            result = session.run("""
                MATCH (h:Hadith {number: $num})-[:HAS_CHAIN]->(first:Person)
                MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
                WHERE ALL(r IN relationships(path) WHERE r.hadith = $num)
                  AND NOT EXISTS((last)-[:NARRATED_FROM {hadith: $num}]->())
                RETURN [node in nodes(path) | node.name] as chain
                LIMIT 1
            """, num=hadith_num)

            for record in result:
                chain_str = ' → '.join(filter(None, record['chain']))
                print(f"   Hadith {hadith_num}: {chain_str[:80]}...")

        # Totals
        print(f"\n📈 Database Totals:")
        result = session.run("MATCH (h:Hadith) RETURN count(h) as count")
        print(f"   Hadiths: {result.single()['count']}")

        result = session.run("MATCH (n:Person) RETURN count(n) as count")
        print(f"   Narrators: {result.single()['count']}")

        result = session.run("MATCH ()-[r:NARRATED_FROM]->() RETURN count(r) as count")
        print(f"   Transmission edges: {result.single()['count']}")

final_validation_optimized()

FINAL GRAPH VALIDATION

📊 Sample Chain Lengths (first 100 hadiths):
   Average: 4.5
   Range: 2 - 14

👥 Top 10 Narrators by Total Connections:
   الزُّهْرِيِّ: 2646 connections (الفقيه الحافظ متفق على جلالته وإتقانه)
   شُعْبَةُ: 1674 connections (ثقة حافظ متقن عابد)
   مَالِكٌ: 1315 connections (رأس المتقنين وكبير المتثبتين)
   عُرْوَةَ بْنَ الزُّبَيْرِ: 1272 connections (ثقة فقيه مشهور)
   أَبِي هُرَيْرَةَ: 1258 connections (صحابي)
   أَنَسٌ: 1024 connections (صحابي)
   عَائِشَةُ: 989 connections (صحابي)
   ابْنِ عُمَرَ: 956 connections (صحابي)
   اللَّيْثُ: 903 connections (ثقة ثبت فقيه إمام مشهور)
   لِابْنِ عَبَّاسٍ: 875 connections (صحابي)

🔗 Sample Chains:
   Hadith 1: الْحُمَيْدِيُّ → سُفْيَانُ → يَحْيَى → مُحَمَّدِ بْنِ إِبْرَاهِيمَ → وَعَلْقَمَة...
   Hadith 2: عَبْدُ اللَّهِ بْنُ يُوسُفَ → مَالِكٌ → هِشَامٍ → عُرْوَةَ بْنَ الزُّبَيْرِ → عَ...
   Hadith 50: إِبْرَاهِيمُ بْنُ حَمْزَةَ → إِبْرَاهِيمُ بْنُ سَعْدٍ → صَالِحٍ → الزُّهْرِيِّ →...
   Hadith 100: آدَمُ → شُعْبَةُ → عَ

In [83]:
def load_to_neo4j_fixed(hadiths_file='bukhari_hadiths.json'):
    """Load scraped hadiths into Neo4j - FIXED VERSION"""

    driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

    with open(hadiths_file, 'r', encoding='utf-8') as f:
        hadiths = json.load(f)

    with driver.session() as session:

        # First, create all narrator nodes from narrator_details
        print("Creating narrator nodes...")
        for hadith in hadiths:
            for narrator in hadith['narrator_details']:
                session.run("""
                    MERGE (n:Person {id: $id})
                    SET n.name = $name,
                        n.fame = $fame,
                        n.rank = $rank,
                        n.is_narrator = true
                """,
                id=narrator['id'],
                name=narrator['name'],
                fame=narrator['fame'],
                rank=narrator['rank']
                )

        print("Creating hadiths and chains...")
        for idx, hadith in enumerate(hadiths):
            if idx % 100 == 0:
                print(f"  Processed {idx}/{len(hadiths)}...")

            # Create hadith node
            session.run("""
                MERGE (h:Hadith {number: $number})
                SET h.matn = $matn,
                    h.full_text = $full_text
            """,
            number=hadith['hadith_number'],
            matn=hadith['matn'],
            full_text=hadith['full_text']
            )

            chain = hadith['chain']

            if not chain:
                continue

            # Create NARRATED_FROM edges between consecutive narrators
            for i in range(len(chain) - 1):
                narrator1_id = chain[i]['id']
                narrator2_id = chain[i + 1]['id']

                # Skip if same ID (self-loop bug)
                if narrator1_id == narrator2_id:
                    continue

                session.run("""
                    MATCH (n1:Person {id: $id1})
                    MATCH (n2:Person {id: $id2})
                    MERGE (n1)-[:NARRATED_FROM {hadith: $hadith_num}]->(n2)
                """,
                id1=narrator1_id,
                id2=narrator2_id,
                hadith_num=hadith['hadith_number']
                )

            # Link hadith to FIRST narrator in chain only
            session.run("""
                MATCH (h:Hadith {number: $hadith_num})
                MATCH (n:Person {id: $narrator_id})
                MERGE (h)-[:HAS_CHAIN]->(n)
            """,
            hadith_num=hadith['hadith_number'],
            narrator_id=chain[0]['id']
            )

    driver.close()
    print(f"✅ Loaded {len(hadiths)} hadiths to Neo4j")

# Clear database and reload
def reset_and_reload():
    driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

    print("Clearing database...")
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")

    driver.close()
    print("Database cleared.")

    # Reload with fixed function
    load_to_neo4j_fixed()

reset_and_reload()

Clearing database...
Database cleared.
Creating narrator nodes...
Creating hadiths and chains...
  Processed 0/7031...
  Processed 100/7031...
  Processed 200/7031...
  Processed 300/7031...
  Processed 400/7031...
  Processed 500/7031...
  Processed 600/7031...
  Processed 700/7031...
  Processed 800/7031...
  Processed 900/7031...
  Processed 1000/7031...
  Processed 1100/7031...
  Processed 1200/7031...
  Processed 1300/7031...
  Processed 1400/7031...
  Processed 1500/7031...
  Processed 1600/7031...
  Processed 1700/7031...
  Processed 1800/7031...
  Processed 1900/7031...
  Processed 2000/7031...
  Processed 2100/7031...
  Processed 2200/7031...
  Processed 2300/7031...
  Processed 2400/7031...
  Processed 2500/7031...
  Processed 2600/7031...
  Processed 2700/7031...
  Processed 2800/7031...
  Processed 2900/7031...
  Processed 3000/7031...
  Processed 3100/7031...
  Processed 3200/7031...
  Processed 3300/7031...
  Processed 3400/7031...
  Processed 3500/7031...
  Processed 360